# Day 23 — nn.Module & the Training Loop, Visualized

The five-step loop from Day 22, now with nn.Module + an optimizer. Train a classifier on
two blobs and watch the decision boundary form.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

c0 = torch.randn(100, 2) * 0.5 + torch.tensor([-2.0, -2.0])
c1 = torch.randn(100, 2) * 0.5 + torch.tensor([2.0, 2.0])
X = torch.cat([c0, c1]); y = torch.cat([torch.zeros(100), torch.ones(100)]).long()

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
opt = torch.optim.Adam(model.parameters(), lr=0.05); crit = nn.CrossEntropyLoss()
losses = []
for _ in range(150):
    opt.zero_grad(); loss = crit(model(X), y); loss.backward(); opt.step()
    losses.append(loss.item())
acc = (model(X).argmax(1) == y).float().mean().item()
print(f'final loss {losses[-1]:.3f},  accuracy {acc:.1%}')

## Loss curve and learned decision boundary

In [ ]:
import matplotlib.pyplot as plt, numpy as np

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
axL.plot(losses); axL.set_title('loss'); axL.set_xlabel('epoch'); axL.grid(alpha=0.3)

xx, yy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad(): zz = model(grid).argmax(1).numpy().reshape(xx.shape)
axR.contourf(xx, yy, zz, alpha=0.3, cmap='coolwarm')
axR.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=12, edgecolor='k', linewidth=0.2)
axR.set_title('decision boundary'); plt.tight_layout(); plt.show()

## Takeaways

- nn.Module + optimizer + loss = the same 5 steps as Day 22, less bookkeeping.
- The network carved a boundary between the classes purely from the loss signal.

**Now build** build_model / train / accuracy in `homework.py`, then `pytest -q`.